# Bunny ICP

In the lecture, we learned about the Iterative Closest Point (ICP) algorithm. In this exercise, you will implement the ICP algorithm to solve the standard Stanford Bunny problem!

**Learning Objectives**
- Understand and Implement the ICP algorithm

**What You'll Implement**
- The ```least_squares_transform``` function to optimize transformation given correspondence
- The ```icp``` algorithm.


## Setup and Imports

Let us first import our standard drake functionality

In [389]:
import numpy as np
from pydrake.all import (PointCloud, Rgba, RigidTransform, RotationMatrix, StartMeshcat, MathematicalProgram, Solve)
from scipy.spatial import KDTree
from functools import partial
from manipulation import FindResource
from manipulation.exercises.grader import Grader
from manipulation.exercises.pose.test_icp import TestICP

In [390]:
# Start the visualizer.
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7004


Let's first visualize the point clouds of Stanford bunny in meshcat!

In [391]:
# Visualize Stanford Bunny
xyzs = np.load(FindResource("models/bunny/bunny.npy"))
cloud = PointCloud(xyzs.shape[1])
cloud.mutable_xyzs()[:] = xyzs

# Pose for the blue bunny
X_blue = RigidTransform(RotationMatrix.MakeXRotation(np.pi / 6), [-0.1, 0.1, 0.1]) # type: ignore

pointcloud_model = xyzs
pointcloud_scene = X_blue.multiply(xyzs)

meshcat.Delete()
meshcat.SetProperty("/Background", "visible", False)
meshcat.SetProperty("/Cameras/default/rotated/<object>", "zoom", 10.5)
meshcat.SetObject("red_bunny", cloud, point_size=0.01, rgba=Rgba(1.0, 0, 0))
meshcat.SetTransform("red_bunny", RigidTransform())
meshcat.SetObject("blue_bunny", cloud, point_size=0.01, rgba=Rgba(0, 0, 1.0))
meshcat.SetTransform("blue_bunny", X_blue)

## Part 1: Point cloud registration with known correspondences

In this section, you will follow the [derivation](http://manipulation.csail.mit.edu/pose.html#registration) to solve the optimization problem below.

$$\begin{aligned} \min_{p\in\mathbb{R}^3,R\in\mathbb{R}^{3\times3}} \quad & \sum_{i=1}^{N_s} \| p + R \hspace{.1cm} {^Op^{m_{c_i}}} - p^{s_i}\|^2, \\ s.t. \quad & RR^T = I, \quad \det(R)=1\end{aligned}$$

The goal is to find the transform that registers the point clouds of the model and the scene, assuming the correspondence is known.  You may refer to the implementation from [deepnote](https://github.com/RussTedrake/manipulation/blob/master/book/pose/icp.ipynb) and the explanation from [textbook](http://manipulation.csail.mit.edu/pose.html#icp).

**YOUR TASK**: In the cell below, implement the ```least_squares_transform``` nethod.

In [ ]:
# def least_squares_transform(scene: np.ndarray, model: np.ndarray) -> RigidTransform:
#     """
#     Calculates the least-squares best-fit transform that maps corresponding
#     points scene to model.
#     Args:
#       scene: 3xN numpy array of corresponding points
#       model: 3xM numpy array of corresponding points
#     Returns:
#       X_BA: A RigidTransform object that maps point_cloud_A on to point_cloud_B
#             such that
#                         X_BA.multiply(model) ~= scene,
#     """
#     X_BA = RigidTransform()
#     # TODO: Implement this function.
#     p_scene = scene.T
#     # print(p_scene.shape)
#     p_model = model.T
#     # print(p_model.shape)  
#     p_s_center = p_scene.mean(axis=0)
#     # print(p_s_center)
#     p_m_center = p_model.mean(axis=0)

#     p_scene_local = p_scene - p_s_center
#     p_model_local = p_model - p_m_center

#     A = p_scene_local.T @ p_model_local
#     # print(A.shape)
#     U, delta, V_T = np.linalg.svd(A)
    
#     R = U @ V_T

#     if np.linalg.det(R) < 0:
#         V_T[-1,:] *= -1
#         R = U @ V_T

#     p = p_s_center - R @ p_m_center
#     X_BA = RigidTransform(RotationMatrix(R), p)

#     return X_BA

In [ ]:
# 定义旋转矩阵从 a, b 的转换函数（3x3，绕 z 轴旋转）
def ab_to_rotation(a_val, b_val):
    return np.array([
        [a_val, -b_val, 0],
        [b_val, a_val, 0],
        [0, 0, 1]
    ])


def least_squares_transform(scene: np.ndarray, model: np.ndarray) -> RigidTransform:
    """
    Calculates the least-squares best-fit transform that maps corresponding
    points scene to model.
    Args:
      scene: 3xN numpy array of corresponding points
      model: 3xM numpy array of corresponding points
    Returns:
      X_BA: A RigidTransform object that maps point_cloud_A on to point_cloud_B
            such that
                        X_BA.multiply(model) ~= scene,
    """
    if scene.shape[1] != model.shape[1]:
        raise ValueError(f"scene shape: {scene.shape[1]} is not equal to model: {model.shape[1]}.")
    
    p_scene = scene  # (3, N)
    p_model = model  # (3, N)
    N = p_scene.shape[1]
    
    # 设置 MathematicalProgram（二次优化）
    prog = MathematicalProgram()
    
    # 优化变量：3D 平移 p 和旋转参数 a, b (a=cos(theta), b=sin(theta))
    p = prog.NewContinuousVariables(3, "p")
    a = prog.NewContinuousVariables(1, "a")[0]  # a = cos(theta)
    b = prog.NewContinuousVariables(1, "b")[0]  # b = sin(theta)
    
    # 添加二次约束：a^2 + b^2 == 1
    prog.AddConstraint(a**2 + b**2 == 1)
    
    # 向量化总成本函数（单个 AddCost，避免 N 次循环）
    def total_squared_distance(vars):
        p_var, a_var, b_var = np.split(vars, [3, 4])
        R = np.array([[a_var[0], -b_var[0], 0], [b_var[0], a_var[0], 0], [0, 0, 1]])
        # 向量化变换：(3,N) = p_var[:,None] + R @ p_model
        transformed = p_var[:, None] + R @ p_model
        err = transformed - p_scene  # (3,N)
        return np.sum(err**2)  # 总 \|err\|^2 (scalar)
    
    # 为每个对应点添加二次成本
    vars_all = np.concatenate([p, [a], [b]])  # 总变量 (5D)

    prog.AddCost(total_squared_distance, vars_all)  # 单个成本！
    
    # 设置初始猜测（identity 变换，避免局部最小）
    prog.SetInitialGuess(p, np.zeros(3))
    prog.SetInitialGuess(a, 1.0)  # cos(0) = 1
    prog.SetInitialGuess(b, 0.0)  # sin(0) = 0
    
    # 求解二次优化问题
    result = Solve(prog)
    if not result.is_success():
        raise RuntimeError("Optimization failed to converge.")
    
    # 提取解并构造 RigidTransform
    p_sol = result.GetSolution(p)  # 3D 平移
    a_sol = result.GetSolution(a)  # cos(theta)
    b_sol = result.GetSolution(b)  # sin(theta)
    # 归一化以防数值误差（可选，但推荐）
    norm = np.sqrt(a_sol**2 + b_sol**2)
    if norm != 0:
        a_sol /= norm
        b_sol /= norm
    R_sol = ab_to_rotation(a_sol, b_sol)  # 转换为旋转矩阵
    
    X_BA = RigidTransform(RotationMatrix(R_sol), p_sol)

    return X_BA

## Part 2: Point correspondence from closest point
The ```least_squares_transform``` function assumes that the point correspondence is known. Unfortunately, this is often not the case, so we will have to estimate the point correspondence as well. A common heuristics for estimating point correspondence is the closest point/nearest neighbor.

We have implemented the closest neighbors using [scipy's implementation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.KDTree.html) of [k-d trees](https://en.wikipedia.org/wiki/K-d_tree).

In [401]:
def nearest_neighbors(
    scene: np.ndarray, model: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """
    Find the nearest (Euclidean) neighbor in model for each
    point in scene
    Args:
        scene: 3xN numpy array of points
        model: 3xM numpy array of points
    Returns:
        distances: (N, ) numpy array of Euclidean distances from each point in
            scene to its nearest neighbor in model.
        indices: (N, ) numpy array of the indices in model of each
            scene point's nearest neighbor - these are the c_i's
    """
    kdtree = KDTree(model.T)

    distances, indices = kdtree.query(scene.T, k=1)

    return distances.flatten(), indices.flatten()

## Part 3: Iterative Closest Point (ICP)
Now you should be able to register two point clouds iteratively by first finding/updating the estimate of point correspondence with ```nearest_neighbors``` and then computing the transform using ```least_squares_transform```. You may refer to the explanation from [textbook](http://manipulation.csail.mit.edu/pose.html#icp).

**YOUR TASK**: In the cell below, complete the implementation of ICP algorithm using the  ```nearest_neighbors``` and ```least_squares_transform``` methods from above.

In [402]:
def icp(
    scene: np.ndarray,
    model: np.ndarray,
    max_iterations: int = 20,
    tolerance: float = 1e-3,
) -> tuple[RigidTransform, float, int]:
    """
    Perform ICP to return the correct relative transform between two set of points.
    Args:
        scene: 3xN numpy array of points
        model: 3xM numpy array of points
        max_iterations: max amount of iterations the algorithm can perform.
        tolerance: tolerance before the algorithm converges.
    Returns:
      X_BA: A RigidTransform object that maps point_cloud_A on to point_cloud_B
            such that
                        X_BA.multiply(model) ~= scene,
      mean_error: Mean of all pairwise distances.
      num_iters: Number of iterations it took the ICP to converge.
    """
    X_BA = RigidTransform()
    X_BA = RigidTransform(RotationMatrix.MakeXRotation(np.pi/6))
    # print(X_BA)
    mean_error = 0
    num_iters = 0
    prev_error = 0

    while True:
        num_iters += 1
        transformed_model = X_BA @ model
        distance, indices = nearest_neighbors(scene, transformed_model)
        # print(indices)
        # indices = range(indices[0])
        # print(indices.shape)
        # TODO: Implement this function
        X_add = least_squares_transform(scene=scene, model=transformed_model[:,indices])
        # print(X_BA)
        X_BA = X_add @ X_BA  # 累计变换
        # 正确计算 mean_error
        corresponding_model = transformed_model[:, indices]  # (3, N)
        corresponding_coor = X_add @ corresponding_model  # (3, N)
        errors = np.linalg.norm(scene - corresponding_coor, axis=0)
        mean_error = np.mean(errors)  # TODO: Modify to add mean error.
        #############

        if abs(mean_error - prev_error) < tolerance or num_iters >= max_iterations:
            break

        prev_error = mean_error
        # meshcat.SetTransform("red_bunny", X_BA)

    return X_BA, mean_error, num_iters


Now you should be able to visualize the registration of the Stanford bunny! Have fun!

In [413]:
import time
meshcat.SetTransform("red_bunny", RigidTransform())
for i in range(1, 20, 4):
    # time.sleep(1/(i*10))
    X_BA, mean_error, num_iters = icp(pointcloud_scene, pointcloud_model, max_iterations=i, tolerance=1e-5)
    meshcat.SetTransform("red_bunny", X_BA)


You may use the grader below to check your implementations

In [414]:
Grader.grade_output([TestICP], [locals()], "results.json")
Grader.print_test_results("results.json")

Total score is 0/6.

Score for Test icp implementation is 0/3.
- Test Failed: Exception while evaluating SNOPT costs and constraints: 'TimeoutError: 'Timed Out'

At:
  /opt/anaconda3/lib/python3.13/site-packages/timeout_decorator/timeout_decorator.py(45): _raise_exception
  /opt/anaconda3/lib/python3.13/site-packages/timeout_decorator/timeout_decorator.py(69): handler
  /opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py(2344): _sum_dispatcher
  /var/folders/k4/mg5n76b53w31r9_hw85j51xh0000gn/T/ipykernel_45634/317226909.py(50): total_squared_distance
  /var/folders/k4/mg5n76b53w31r9_hw85j51xh0000gn/T/ipykernel_45634/317226909.py(62): least_squares_transform
  /var/folders/k4/mg5n76b53w31r9_hw85j51xh0000gn/T/ipykernel_45634/875953354.py(36): icp
  /opt/anaconda3/lib/python3.13/site-packages/manipulation/exercises/pose/test_icp.py(87): test_icp
  /opt/anaconda3/lib/python3.13/site-packages/timeout_decorator/timeout_decorator.py(82): new_function
  /opt/anaconda3/lib/pyt

## VERIFICATION IN GRADESCOPE

**Prerequisites:** You must complete ALL the TODOs above before these verification exercises will work!

**Instructions:** Complete the exercises that follow and upload your solutions to Gradescope


## Verification 1: Least Squares Transforms

**Question:** What does the `X_BA` rotation matrix returned from ICP above correspond to? Select the appropraite answer in Gradescope.

A. A +60° rotation about the z-axis 
B. A +30° rotation about the y-axis 
C. A -30° rotation about the x-axis 
D. A +30° rotation about the x-axis

**HINT**: Use the `RotationMatrix.MakeX/Y/ZRotation()` methods to create rotation matrices. 